## Installation of LLama3

In [19]:
!apt-get update
!apt-get install -y zstd


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [20]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [21]:
!nohup ollama serve > ollama.log 2>&1 &


In [22]:
import time
time.sleep(5)
!which ollama


/usr/local/bin/ollama


#### Saving llama3 locally temporarily

In [23]:

!pip install ollama
!ollama pull llama3



## Getting user description

In [202]:
problem_text = ("""31. Next Permutation
Attempted
Medium
Topics
premium lock icon
Companies
A permutation of an array of integers is an arrangement of its members into a sequence or linear order.

For example, for arr = [1,2,3], the following are all the permutations of arr: [1,2,3], [1,3,2], [2, 1, 3], [2, 3, 1], [3,1,2], [3,2,1].
The next permutation of an array of integers is the next lexicographically greater permutation of its integer. More formally, if all the permutations of the array are sorted in one container according to their lexicographical order, then the next permutation of that array is the permutation that follows it in the sorted container. If such arrangement is not possible, the array must be rearranged as the lowest possible order (i.e., sorted in ascending order).

For example, the next permutation of arr = [1,2,3] is [1,3,2].
Similarly, the next permutation of arr = [2,3,1] is [3,1,2].
While the next permutation of arr = [3,2,1] is [1,2,3] because [3,2,1] does not have a lexicographical larger rearrangement.
Given an array of integers nums, find the next permutation of nums.

The replacement must be in place and use only constant extra memory.



Example 1:

Input: nums = [1,2,3]
Output: [1,3,2]
Example 2:

Input: nums = [3,2,1]
Output: [1,2,3]
Example 3:

Input: nums = [1,1,5]
Output: [1,5,1]


Constraints:

1 <= nums.length <= 100
0 <= nums[i] <= 100

class Solution:
    def nextPermutation(self, nums: List[int]) -> None:
        """)


In [287]:
problem_text=("""34. Find First and Last Position of Element in Sorted Array
Attempted
Medium
Topics
premium lock icon
Companies
Given an array of integers nums sorted in non-decreasing order, find the starting and ending position of a given target value.

If target is not found in the array, return [-1, -1].

You must write an algorithm with O(log n) runtime complexity.



Example 1:

Input: nums = [5,7,7,8,8,10], target = 8
Output: [3,4]
Example 2:

Input: nums = [5,7,7,8,8,10], target = 6
Output: [-1,-1]
Example 3:

Input: nums = [], target = 0
Output: [-1,-1]


Constraints:

0 <= nums.length <= 105
-109 <= nums[i] <= 109
nums is a non-decreasing array.
-10
class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
      """)

In [226]:
problem_text= ("""53. Maximum Subarray
Medium
Topics
premium lock icon
Companies
Given an integer array nums, find the subarray with the largest sum, and return its sum.



Example 1:

Input: nums = [-2,1,-3,4,-1,2,1,-5,4]
Output: 6
Explanation: The subarray [4,-1,2,1] has the largest sum 6.
Example 2:

Input: nums = [1]
Output: 1
Explanation: The subarray [1] has the largest sum 1.
Example 3:

Input: nums = [5,4,-1,7,8]
Output: 23
Explanation: The subarray [5,4,-1,7,8] has the largest sum 23.


Constraints:

1 <= nums.length <= 105
-104 <= nums[i] <= 104

solution: class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
 """)

## Preprocessing the input to extract problem description, constraints, solution if any

In [288]:
import ollama
import json
import re

MODEL_NAME = "llama3"


# =========================
# JSON UTILS
# =========================
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else ""


def safe_parse(text):
    try:
        return json.loads(text)
    except:
        return None


# =========================
# DESCRIPTION PASS (LLM)
# =========================
def parse_description(text):
    prompt = f"""
Extract the problem description from the input.

Return STRICT JSON:
{{ "description": "" }}

RULES:
- ONLY JSON output
- DO NOT hallucinate or add new facts
- Extract ONLY from INPUT
- You MAY clean formatting noise like:
  - "Solved", "Medium", "Topics", etc.
- Preserve:
  - all problem statements
  - all examples
- EXCLUDE:
  - constraints section
  - code

CRITICAL:
- If unsure, still extract best possible description
- Never return null or empty unless input is empty

INPUT START
{text}
INPUT END
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 600}
    )

    parsed = safe_parse(extract_json(res["message"]["content"]))
    return parsed["description"] if parsed and "description" in parsed else ""

# =========================
# CONSTRAINTS PASS (LLM)
# =========================
def parse_constraints(text):
    prompt = f"""
Extract OR infer constraints.

Return STRICT JSON:
{{ "constraints": "" }}

RULES:
- If constraints exist → copy EXACTLY
- If missing → infer realistic constraints
- Format:
  1 <= n <= 10^5
  0 <= arr[i] <= 10^4
- No explanations

-------------------------
FEW SHOTS
-------------------------

INPUT:
Constraints:
1 <= strs.length <= 200
0 <= strs[i].length <= 200

OUTPUT:
{{ "constraints": "1 <= strs.length <= 200\\n0 <= strs[i].length <= 200" }}

-------------------------

INPUT:
Constraints:
3 <= nums.length <= 3000
-10^5 <= nums[i] <= 10^5

OUTPUT:
{{ "constraints": "3 <= nums.length <= 3000\\n-10^5 <= nums[i] <= 10^5" }}

-------------------------

INPUT:
3Sum Closest

OUTPUT:
{{ "constraints": "3 <= nums.length <= 500\\n-1000 <= nums[i] <= 1000\\n-10^4 <= target <= 10^4" }}

-------------------------
INPUT:
{text}
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 150}
    )

    parsed = safe_parse(extract_json(res["message"]["content"]))
    return parsed["constraints"] if parsed else ""


# =========================
# SOLUTION PASS (REGEX — RELIABLE)
# =========================
def parse_solution(text):
    """
    Extract FIRST class Solution block reliably
    """

    pattern = r"class Solution:\n(?:\s+.*\n?)*"
    match = re.search(pattern, text)

    if match:
        return match.group(0).rstrip()

    return ""


# =========================
# MAIN PIPELINE
# =========================
def structured_preprocess(text):

    description = parse_description(text)
    constraints = parse_constraints(text)
    solution = parse_solution(text)   # <-- DO NOT JSON PARSE

    return {
        "description": description,
        "constraints": constraints,
        "solution": solution
    }


# =========================
# RUN
# =========================
structured_data = structured_preprocess(problem_text)
print(structured_data)


{'description': 'Find First and Last Position of Element in Sorted Array\nGiven an array of integers nums sorted in non-decreasing order, find the starting and ending position of a given target value.\nIf target is not found in the array, return [-1, -1].\nYou must write an algorithm with O(log n) runtime complexity.', 'constraints': '0 <= nums.length <= 10^5\n-10^9 <= nums[i] <= 10^9', 'solution': 'class Solution:\n    def searchRange(self, nums: List[int], target: int) -> List[int]:'}


In [289]:
print("description:\n", structured_data["description"])
description = structured_data["description"]
print("\nconstraints:\n", structured_data["constraints"])
constraints= structured_data["constraints"]
print("\nsolution:\n", structured_data["solution"])
solution = structured_data["solution"]


description:
 Find First and Last Position of Element in Sorted Array
Given an array of integers nums sorted in non-decreasing order, find the starting and ending position of a given target value.
If target is not found in the array, return [-1, -1].
You must write an algorithm with O(log n) runtime complexity.

constraints:
 0 <= nums.length <= 10^5
-10^9 <= nums[i] <= 10^9

solution:
 class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:


## getting sample test cases

In [290]:
import re
import ast


# =========================
# SAMPLE TEST EXTRACTION (FIXED)
# =========================
def extract_samples(text):
    """
    Extracts (input, output) pairs from coding problem text.
    Properly stops at Explanation to avoid leakage.
    """

    text = text.replace("\r", "")

    # ✅ FIX: stop at Explanation as well
    pattern = r"Input:\s*(.*?)\n\s*Output:\s*(.*?)(?=\n\s*(Explanation:|Input:|Example|Constraints|$))"

    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)

    samples = []
    for inp, out, _ in matches:
        samples.append({
            "input": inp.strip(),
            "output": out.strip()
        })

    return samples


# =========================
# INPUT PARSER (GENERIC ENOUGH FOR NOW)
# =========================
def parse_input_string(input_str):
    """
    Keeps raw structure + lightweight extraction.
    (Main execution still uses exec-based parser elsewhere)
    """

    arrays = re.findall(r"\[.*?\]", input_str)
    nums = re.findall(r"-?\d+", input_str)

    return {
        "arrays": arrays,
        "numbers": list(map(int, nums)) if nums else []
    }


# =========================
# OUTPUT PARSER (SIMPLIFIED + ROBUST)
# =========================
def parse_output(output_str):
    """
    Clean parser assuming extraction is correct.
    """

    output_str = output_str.strip()

    # ✅ try exact structured parse
    try:
        return ast.literal_eval(output_str)
    except:
        pass

    # ✅ fallback: single number
    nums = re.findall(r"-?\d+", output_str)
    if len(nums) == 1:
        return int(nums[0])

    # fallback raw
    return output_str


# =========================
# PIPELINE WRAPPER
# =========================
def build_testcases(problem_text):
    """
    End-to-end sample extraction pipeline.
    """

    raw_samples = extract_samples(problem_text)

    testcases = []

    for s in raw_samples:
        inp = parse_input_string(s["input"])
        out = parse_output(s["output"])

        testcases.append({
            "input_raw": s["input"],
            "input_parsed": inp,
            "expected": out
        })

    return testcases


# =========================
# EXAMPLE USAGE
# =========================
tests = build_testcases(problem_text)

print(tests)

[{'input_raw': 'nums = [5,7,7,8,8,10], target = 8', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 8]}, 'expected': [3, 4]}, {'input_raw': 'nums = [5,7,7,8,8,10], target = 6', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 6]}, 'expected': [-1, -1]}, {'input_raw': 'nums = [], target = 0', 'input_parsed': {'arrays': ['[]'], 'numbers': [0]}, 'expected': [-1, -1]}]


## Creating working solution for the description and testing it on the user provided test cases.

In [291]:
import ollama
import ast
import traceback

MODEL_NAME = "llama3"


# =========================
# CLEAN CODE
# =========================
def clean_code(text):
    text = text.replace("```python", "").replace("```", "")

    idx = text.find("class Solution")
    if idx != -1:
        text = text[idx:]

    return text.strip()


# =========================
# LLM GENERATION
# =========================
def llm_generate_solution(description, constraints, solution_template, error_msg=None):
    prompt = f"""
You are a Python competitive programming engine.

STRICT RULES:
- Output ONLY executable Python code
- NO explanations
- MUST define class Solution
- MUST match function signature exactly
- ONLY complete the function body

TEMPLATE:
{solution_template}

PROBLEM:
{description}

CONSTRAINTS:
{constraints}
"""

    if error_msg:
        prompt += f"\n\nPREVIOUS ERROR:\n{error_msg}\nFix the code."

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return clean_code(res["message"]["content"])


# =========================
# PARSE INPUTS (FIXED)
# =========================
def parse_inputs(input_raw):
    env = {}

    # split ONLY on top-level commas (safe enough for LC-style inputs)
    parts = []
    current = ""
    bracket_level = 0

    for ch in input_raw:
        if ch == ',' and bracket_level == 0:
            parts.append(current.strip())
            current = ""
        else:
            current += ch
            if ch in "[{(":
                bracket_level += 1
            elif ch in "]})":
                bracket_level -= 1

    if current:
        parts.append(current.strip())

    # now execute each assignment separately
    for part in parts:
        if "=" not in part:
            continue
        exec(part, {}, env)

    return list(env.values())

# =========================
# GET FUNCTION (GENERIC)
# =========================
def get_function(sol, num_args):
    candidates = []

    for name in dir(sol):
        if name.startswith("__"):
            continue

        fn = getattr(sol, name)

        if callable(fn):
            try:
                if fn.__code__.co_argcount - 1 == num_args:
                    candidates.append((name, fn))
            except:
                continue

    if not candidates:
        raise Exception("No matching function found")

    # ✅ Prefer method that does NOT look like a helper
    # (heuristic: avoid names starting with 'find', 'helper', etc.)
    priority = []
    fallback = []

    for name, fn in candidates:
        if name.lower().startswith(("find", "helper", "dfs", "bfs", "util")):
            fallback.append(fn)
        else:
            priority.append(fn)

    if priority:
        return priority[0]

    return fallback[0]


# =========================
# RUN SINGLE TEST
# =========================
def run_once(code, test):
    try:
        env = {}

        # ensure typing import if needed
        if "List[" in code and "from typing import" not in code:
            code = "from typing import List\n" + code

        exec(code, env)

        import copy

        sol = env["Solution"]()

        args = parse_inputs(test["input_raw"])
        args_copy = copy.deepcopy(args)  # optional but useful for debugging later

        func = get_function(sol, len(args))

        result = func(*args)

        # handle in-place modification functions
        if result is None:
            if len(args) == 1:
                result = args[0]
            else:
                result = args

        return {
            "status": "OK",
            "output": result,
            "error": None,
            "code": code
        }

    except Exception:
        return {
            "status": "ERROR",
            "output": None,
            "error": traceback.format_exc(),
            "code": code
        }

# =========================
# GENERATE ONCE + TEST ALL + RETRY
# =========================
def solve_with_retry(description, constraints, solution_template, tests, max_attempts=3):
    last_error = None
    best_results = None
    best_score = -1
    best_code = None

    for _ in range(max_attempts):
        code = llm_generate_solution(description, constraints, solution_template, last_error)

        results = []
        passed = 0

        for i, t in enumerate(tests):
            res = run_once(code, t)

            is_match = (
                res["status"] == "OK" and
                res["output"] == t["expected"]
            )

            if is_match:
                passed += 1
            else:
                last_error = res["error"]

            results.append({
                "test_id": i,
                "input": t["input_raw"],
                "expected": t["expected"],
                "output": res["output"],
                "status": res["status"],
                "error": res["error"],
                "match": is_match,
                "solution_used": code
            })

        # keep best attempt
        if passed > best_score:
            best_score = passed
            best_results = results
            best_code = code

        # early stop if perfect
        if passed == len(tests):
            break

    return best_code, best_results


# =========================
# MAIN EXECUTION (YOUR FLOW)
# =========================
results = []

code, results = solve_with_retry(
    description=description,
    constraints=constraints,
    solution_template=solution,
    tests=tests,
    max_attempts=3
)


# =========================
# SUMMARY
# =========================
passed = sum(1 for r in results if r["match"])
errors = sum(1 for r in results if r["status"] == "ERROR")
total = len(results)

print(f"\nPASS RATE: {passed}/{total}")
print(f"ERRORS: {errors}/{total}\n")

for r in results:
    print("----")
    print("TEST:", r["input"])
    print("EXPECTED:", r["expected"])
    print("GOT:", r["output"])
    print("STATUS:", r["status"])
    print("MATCH:", r["match"])

    if not r["match"]:
        print("ERROR:", r["error"])

    print("\nSOLUTION USED:\n", r["solution_used"])


print("\nFINAL BEST CODE:\n")
print(code)


PASS RATE: 3/3
ERRORS: 0/3

----
TEST: nums = [5,7,7,8,8,10], target = 8
EXPECTED: [3, 4]
GOT: [3, 4]
STATUS: OK
MATCH: True

SOLUTION USED:
 class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        return [self.find_first(nums, target), self.find_last(nums, target)]

    def find_first(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            else:
                right = mid
        return left if left < len(nums) and nums[left] == target else -1

    def find_last(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] <= target:
                left = mid + 1
            else:
                right = mid
        return right - 1 if right > 0 and nums[right - 1] == target

## Creating our own edge cases from the constraints for robustness

### we need to satisfy:
1. Boundary testing: checks behavior at minimum, maximum, and edge limits of input constraints
2. Extreme value testing: verifies correctness under very large, very small, or uniform value distributions
3. Pattern testing: validates logic on structured inputs like sorted, reversed, or repeating sequences
4. Random testing: ensures general correctness over unpredictable small-scale inputs
5. Stress testing: evaluates performance and correctness near maximum constraint sizes
6. Deterministic validation: compares exact outputs from candidate and oracle solutions for correctness
7. Failure visibility testing: logs full input-output mismatch details for debugging and analysis
8. Error classification testing: categorizes failures into logic, runtime, or performance issues using rule-based checks

In [292]:
import ollama
import json
import re

MODEL_NAME = "llama3"

TEST_TYPES = [
    "boundary",
    "extreme",
    "pattern",
    "random",
    "stress",
    "deterministic",
    "failure_visibility",
    "error_classification"
]


# =========================
# STRICT JSON PARSER
# =========================
def safe_json_parse(text):

    if not text:
        return None

    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\[\s*\{.*\}\s*\]", text, re.DOTALL)
    if match:
        text = match.group(0)

    try:
        return json.loads(text)
    except:
        return None


# =========================
# STRUCTURE TEMPLATE
# =========================
def build_structure_template(sample_tests):

    if not sample_tests:
        return {}

    return sample_tests[0]["input_parsed"]


# =========================
# RELAXED STRUCTURE MATCH
# =========================
def matches_structure(template, candidate):

    if not isinstance(candidate, dict):
        return False

    # candidate must contain required keys
    for k in template:
        if k not in candidate:
            return False

    return True


# =========================
# VALIDATOR (BALANCED)
# =========================
def validate_tests(tests, sample_tests):

    if not isinstance(tests, list):
        return []

    template = build_structure_template(sample_tests)

    clean = []
    seen = set()

    for t in tests:

        if not isinstance(t, dict):
            continue

        required = ["type", "input_raw", "input_parsed", "expected"]
        if not all(k in t for k in required):
            continue

        # ✅ fix input_raw instead of rejecting
        if not isinstance(t["input_raw"], str):
            t["input_raw"] = str(t["input_raw"])

        raw = t["input_raw"]

        # reject unsafe expressions
        forbidden = ["...", "range(", "len(", "**", "10^"]
        if any(f in raw for f in forbidden):
            continue

        ip = t["input_parsed"]

        if not isinstance(ip, dict):
            continue

        # ✅ relaxed structure check
        if not matches_structure(template, ip):
            continue

        # ❗ DO NOT over-restrict expected type
        # (handled later via execution)

        # deduplicate
        key = json.dumps(t, sort_keys=True)
        if key in seen:
            continue

        seen.add(key)
        clean.append(t)

    return clean


# =========================
# SCORING
# =========================
def score_test(t):

    raw = t.get("input_raw", "")

    if not isinstance(raw, str):
        return 0

    score = 0

    if "[" in raw:
        score += 1

    if len(re.findall(r"-?\d+", raw)) > 3:
        score += 1

    if any(x in raw for x in ["-1", "0", "100", "999"]):
        score += 1

    return score


# =========================
# GENERATE PER TYPE
# =========================
def generate_for_type(description, constraints, test_type, sample_tests, k=5):

    template = build_structure_template(sample_tests)

    prompt = f"""
You generate STRICT JSON test cases.

========================
HARD RULES
========================
- ONLY JSON ARRAY
- EXACTLY {k} test cases
- NO explanations
- NO markdown
- NO "...", range(), len(), expressions
- ALL values must be explicit

========================
INPUT STRUCTURE TEMPLATE
========================
{json.dumps(template, indent=2)}

RULES:
- MUST include all keys from template
- SAME structure (dict/list nesting)
- LIST LENGTHS CAN CHANGE
- ONLY values change

========================
REFERENCE TEST CASES
========================
{json.dumps(sample_tests, indent=2)}

========================
FORMAT
========================
[
  {{
    "type": "{test_type}",
    "input_raw": "...",
    "input_parsed": {{...}},
    "expected": ...
  }}
]

TYPE:
{test_type}

PROBLEM:
{description}

CONSTRAINTS:
{constraints}
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    parsed = safe_json_parse(res["message"]["content"])

    if not parsed:
        print(f"[WARN] JSON parse failed for type: {test_type}")
        return []

    validated = validate_tests(parsed, sample_tests)

    # ✅ soft fallback (not fake)
    if not validated:
        print(f"[WARN] All rejected → using raw for type: {test_type}")
        return parsed[:2] if isinstance(parsed, list) else []

    return validated


# =========================
# MAIN PIPELINE
# =========================
def generate_all_tests(description, constraints, sample_tests):

    final_tests = []

    for test_type in TEST_TYPES:

        candidates = []

        # retry loop
        for _ in range(3):
            batch = generate_for_type(
                description,
                constraints,
                test_type,
                sample_tests,
                k=5
            )

            if batch:
                candidates.extend(batch)

            if len(candidates) >= 3:
                break

        if not candidates:
            print(f"[WARN] No tests generated for type: {test_type}")
            continue

        # rank
        candidates.sort(key=score_test, reverse=True)

        # take best 2
        final_tests.extend(candidates[:2])

    return final_tests


# =========================
# DEBUG
# =========================
def debug_tests(tests):

    print("\n===== FINAL TEST SET =====\n")

    for i, t in enumerate(tests):
        print(f"{i+1}. {t}")


# =========================
# USAGE
# =========================
# IMPORTANT: sample_tests must come from your extractor

# sample_tests = build_testcases(problem_text)

generated_tests = generate_all_tests(
    description,
    constraints,
    tests
)

debug_tests(generated_tests)

[WARN] All rejected → using raw for type: boundary
[WARN] All rejected → using raw for type: boundary
[WARN] All rejected → using raw for type: extreme
[WARN] All rejected → using raw for type: extreme
[WARN] All rejected → using raw for type: pattern
[WARN] All rejected → using raw for type: pattern
[WARN] All rejected → using raw for type: random
[WARN] All rejected → using raw for type: random
[WARN] All rejected → using raw for type: stress
[WARN] All rejected → using raw for type: stress
[WARN] All rejected → using raw for type: deterministic
[WARN] All rejected → using raw for type: deterministic
[WARN] All rejected → using raw for type: failure_visibility
[WARN] All rejected → using raw for type: failure_visibility
[WARN] All rejected → using raw for type: error_classification
[WARN] All rejected → using raw for type: error_classification

===== FINAL TEST SET =====

1. {'input_raw': 'nums = [1,3,4,6,7], target = 3', 'input_parsed': {'arrays': ['[1,3,4,6,7]'], 'numbers': [1, 3, 

## TYPE-2

In [293]:
import ollama
import json
import re

MODEL_NAME = "llama3"

TEST_TYPES = [
    "boundary",
    "extreme",
    "pattern",
    "random",
    "stress",
    "deterministic",
    "failure_visibility",
    "error_classification"
]


# =========================
# STRICT JSON PARSER
# =========================
def safe_json_parse(text):

    if not text:
        return None

    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\[\s*\{.*\}\s*\]", text, re.DOTALL)
    if match:
        text = match.group(0)

    try:
        return json.loads(text)
    except:
        return None


# =========================
# STRUCTURE TEMPLATE
# =========================
def build_structure_template(sample_tests):

    if not sample_tests:
        return {}

    return sample_tests[0]["input_parsed"]


# =========================
# NORMALIZE INPUT (CRITICAL FIX)
# =========================
def normalize_input(ip, template):

    if not isinstance(ip, dict):
        return None

    normalized = {}

    for k in template:

        if k in ip:
            normalized[k] = ip[k]

        # attempt recovery
        elif k == "nums" and "arrays" in ip:
            try:
                normalized[k] = eval(ip["arrays"][0])
            except:
                return None

        elif k == "target" and "numbers" in ip:
            try:
                normalized[k] = ip["numbers"][-1]
            except:
                return None

        else:
            return None  # cannot fix missing key

    return normalized


# =========================
# STRUCTURE MATCH
# =========================
def matches_structure(template, candidate):

    if not isinstance(candidate, dict):
        return False

    for k in template:
        if k not in candidate:
            return False

    return True


# =========================
# VALIDATOR (ROBUST)
# =========================
def validate_tests(tests, sample_tests):

    if not isinstance(tests, list):
        return []

    template = build_structure_template(sample_tests)

    clean = []
    seen = set()

    for t in tests:

        if not isinstance(t, dict):
            continue

        required = ["type", "input_raw", "input_parsed", "expected"]
        if not all(k in t for k in required):
            continue

        # enforce string input
        if not isinstance(t["input_raw"], str):
            t["input_raw"] = str(t["input_raw"])

        raw = t["input_raw"]

        # reject unsafe expressions
        forbidden = ["...", "range(", "len(", "**", "10^"]
        if any(f in raw for f in forbidden):
            continue

        ip = t["input_parsed"]

        if not isinstance(ip, dict):
            continue

        # normalize to template
        ip_fixed = normalize_input(ip, template)

        if not ip_fixed:
            continue

        # enforce structure
        if not matches_structure(template, ip_fixed):
            continue

        t["input_parsed"] = ip_fixed

        # deduplicate
        key = json.dumps(t, sort_keys=True)
        if key in seen:
            continue

        seen.add(key)
        clean.append(t)

    return clean


# =========================
# SCORING
# =========================
def score_test(t):

    raw = t.get("input_raw", "")

    if not isinstance(raw, str):
        return 0

    score = 0

    if "[" in raw:
        score += 1

    if len(re.findall(r"-?\d+", raw)) > 3:
        score += 1

    if any(x in raw for x in ["-1", "0", "100", "999"]):
        score += 1

    return score


# =========================
# GENERATE PER TYPE
# =========================
def generate_for_type(description, constraints, test_type, sample_tests, k=5):

    template = build_structure_template(sample_tests)

    prompt = f"""
You generate STRICT JSON test cases.

RULES:
- ONLY JSON ARRAY
- EXACTLY {k} test cases
- NO explanations
- NO markdown
- NO "...", range(), len()
- ALL values must be explicit

INPUT TEMPLATE:
{json.dumps(template, indent=2)}

RULES:
- MUST include same keys
- SAME structure
- ONLY values change
- LIST lengths can vary

REFERENCE:
{json.dumps(sample_tests, indent=2)}

FORMAT:
[
  {{
    "type": "{test_type}",
    "input_raw": "...",
    "input_parsed": {{...}},
    "expected": ...
  }}
]

TYPE: {test_type}

PROBLEM:
{description}

CONSTRAINTS:
{constraints}
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    parsed = safe_json_parse(res["message"]["content"])

    if not parsed:
        print(f"[WARN] JSON parse failed for type: {test_type}")
        return []

    validated = validate_tests(parsed, sample_tests)

    # soft fallback (structured)
    if not validated:
        print(f"[WARN] All rejected → fallback for type: {test_type}")
        return sample_tests[:2]

    return validated


# =========================
# MAIN PIPELINE
# =========================
def generate_all_tests(description, constraints, sample_tests):

    final_tests = []

    for test_type in TEST_TYPES:

        candidates = []

        for _ in range(3):
            batch = generate_for_type(
                description,
                constraints,
                test_type,
                sample_tests,
                k=5
            )

            if batch:
                candidates.extend(batch)

            if len(candidates) >= 3:
                break

        if not candidates:
            print(f"[WARN] No tests for type: {test_type}")
            continue

        candidates.sort(key=score_test, reverse=True)

        final_tests.extend(candidates[:2])

    return final_tests


# =========================
# DEBUG
# =========================
def debug_tests(tests):

    print("\n===== FINAL TEST SET =====\n")

    for i, t in enumerate(tests):
        print(f"{i+1}. {t}")


# =========================
# USAGE
# =========================
generated_tests_2 = generate_all_tests(
    description,
    constraints,
    tests
)

debug_tests(generated_tests_2)


===== FINAL TEST SET =====

1. {'type': 'boundary', 'input_raw': 'nums = [5,7,7,8,8,10], target = 8', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 8]}, 'expected': [3, 4]}
2. {'type': 'boundary', 'input_raw': 'nums = [5,7,7,8,8,10], target = 6', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 6]}, 'expected': [-1, -1]}
3. {'type': 'extreme', 'input_raw': 'nums = [5,7,7,8,8,10], target = 8', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 8]}, 'expected': [3, 4]}
4. {'type': 'extreme', 'input_raw': 'nums = [5,7,7,8,8,10], target = 6', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 6]}, 'expected': [-1, -1]}
5. {'type': 'pattern', 'input_raw': 'nums = [5,7,7,8,8,10], target = 8', 'input_parsed': {'arrays': ['[5,7,7,8,8,10]'], 'numbers': [5, 7, 7, 8, 8, 10, 8]}, 'expected': [3, 4]}
6. {'type': 'pattern', 'input_raw': 'nums = [1,3,4,6], target = 3', 'input_parsed': {

## optimised code

In [299]:
import json
import re
import typing
import ollama

MODEL_NAME = "llama3"

# =========================
# CLEAN CODE
# =========================
def clean_code(text):
    text = re.sub(r"```python|```", "", text)

    idx = text.find("class Solution")
    if idx != -1:
        text = text[idx:]

    return text.strip()


# =========================
# VALID CODE CHECK
# =========================
def is_valid_code(code):
    return code and "class Solution" in code


# =========================
# RUN ONCE
# =========================
def run_once(code, test):
    try:
        env = {
            "__builtins__": __builtins__,
            "List": typing.List
        }

        exec(code, env, env)

        sol = env["Solution"]()

        # auto detect function
        func = None
        for name in dir(sol):
            if name.startswith("__"):
                continue
            candidate = getattr(sol, name)
            if callable(candidate):
                func = candidate
                break

        inp = test["input_parsed"]

        if "nums" in inp and "target" in inp:
            args = [inp["nums"], inp["target"]]
        elif "numbers" in inp:
            args = [inp["numbers"][:-1], inp["numbers"][-1]]
        elif "arrays" in inp:
            nums = eval(inp["arrays"][0])
            target = inp.get("target", inp.get("numbers", [None])[-1])
            args = [nums, target]
        else:
            raise ValueError(f"Unsupported input: {inp}")

        output = func(*args)

        return {"status": "OK", "output": output}

    except Exception as e:
        return {"status": "ERROR", "output": None, "error": str(e)}


# =========================
# RUN TESTS
# =========================
def run_tests(code, tests):
    results = []

    for i, t in enumerate(tests):
        res = run_once(code, t)

        results.append({
            "test_id": i,
            "input": t["input_raw"],
            "expected": t["expected"],
            "output": res.get("output"),
            "status": res.get("status"),
            "error": res.get("error"),
            "match": res.get("output") == t["expected"]
        })

    return results


# =========================
# FAILED CASES
# =========================
def get_failed_cases(results):
    return [
        {
            "input": r["input"],
            "expected": r["expected"],
            "output": r["output"],
            "error": r["error"]
        }
        for r in results if not r["match"]
    ]


# =========================
# OPTIMIZER
# =========================
def optimize_solution(code, description, constraints, failed_cases):

    prompt = f"""
You are a competitive programming optimizer.

GOAL:
Improve time and space complexity while preserving correctness.

RULES:
- Output ONLY Python code
- Must define class Solution
- No explanations
- Do NOT remove correctness

PROBLEM:
{description}

CONSTRAINTS:
{constraints}

CURRENT CODE:
{code}

FAILED CASES:
{json.dumps(failed_cases, indent=2)}

OUTPUT:
Return ONLY valid Python class Solution.
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return clean_code(res["message"]["content"])


# =========================
# FULL PIPELINE
# =========================
def full_pipeline(base_solution, description, constraints, tests, validated_tests):

    print("\n================ PIPELINE START ================\n")

    if not is_valid_code(base_solution):
        raise ValueError("Invalid base solution")

    print("\nBASE SOLUTION:\n")
    print(base_solution)



    # STEP 2: VALIDATION ON BASE
    val_results_base = run_tests(base_solution, validated_tests)
    val_pass_base = sum(r["match"] for r in val_results_base)

    print(f"BASE VALIDATION: {val_pass_base}/{len(val_results_base)}")

    # STEP 3: FAILURES
    failed_cases = get_failed_cases( val_results_base)

    print("\nFAILED CASES:")
    print(json.dumps(failed_cases, indent=2))

    # STEP 4: OPTIMIZE
    optimized_code = optimize_solution(
        base_solution,
        description,
        constraints,
        failed_cases
    )

    print("\nOPTIMIZED CODE:\n")
    print(optimized_code)

    if not is_valid_code(optimized_code):
        print("❌ Invalid optimized → fallback")
        return base_solution

    # STEP 5: TEST OPTIMIZED
    opt_base_results = run_tests(optimized_code, tests)
    opt_val_results = run_tests(optimized_code, validated_tests)

    opt_base_pass = sum(r["match"] for r in opt_base_results)
    opt_val_pass = sum(r["match"] for r in opt_val_results)

    print(f"\nOPT BASE: {opt_base_pass}/{len(opt_base_results)}")
    print(f"OPT VAL: {opt_val_pass}/{len(opt_val_results)}")

    # STEP 6: ACCEPT ONLY IF NOT WORSE
    if opt_val_pass < val_pass_base:
        print("❌ Optimization worse → revert")
        final_code = base_solution
    else:
        final_code = optimized_code

    print("\n================ FINAL CODE ================\n")
    print(final_code)

    print("\n================ PIPELINE END ================\n")

    return final_code

In [301]:
final_code = full_pipeline(
    base_solution=code,
    description=description,
    constraints=constraints,
    tests=tests,
    validated_tests=generated_tests_2
)


================ PIPELINE START ================


BASE SOLUTION:

class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        return [self.find_first(nums, target), self.find_last(nums, target)]

    def find_first(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            else:
                right = mid
        return left if left < len(nums) and nums[left] == target else -1

    def find_last(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] <= target:
                left = mid + 1
            else:
                right = mid
        return right - 1 if right > 0 and nums[right - 1] == target else -1
BASE VALIDATION: 0/16

FAILED CASES:
[
  {
    "input": "nums = [5

## testing

In [302]:
import json
import re
import typing
import ollama

MODEL_NAME = "llama3"

# =========================
# CLEAN CODE
# =========================
def clean_code(text):
    text = re.sub(r"```python|```", "", text)

    idx = text.find("class Solution")
    if idx != -1:
        text = text[idx:]

    return text.strip()


# =========================
# VALID CODE CHECK
# =========================
def is_valid_code(code):
    return code and "class Solution" in code


# =========================
# RUN ONCE
# =========================
def run_once(code, test):
    try:
        env = {
            "__builtins__": __builtins__,
            "List": typing.List
        }

        exec(code, env, env)

        sol = env["Solution"]()

        # auto detect function
        func = None
        for name in dir(sol):
            if name.startswith("__"):
                continue
            candidate = getattr(sol, name)
            if callable(candidate):
                func = candidate
                break

        inp = test["input_parsed"]

        if "nums" in inp and "target" in inp:
            args = [inp["nums"], inp["target"]]
        elif "numbers" in inp:
            args = [inp["numbers"][:-1], inp["numbers"][-1]]
        elif "arrays" in inp:
            nums = eval(inp["arrays"][0])
            target = inp.get("target", inp.get("numbers", [None])[-1])
            args = [nums, target]
        else:
            raise ValueError(f"Unsupported input: {inp}")

        output = func(*args)

        return {"status": "OK", "output": output}

    except Exception as e:
        return {"status": "ERROR", "output": None, "error": str(e)}


# =========================
# RUN TESTS
# =========================
def run_tests(code, tests):
    results = []

    for i, t in enumerate(tests):
        res = run_once(code, t)

        results.append({
            "test_id": i,
            "input": t["input_raw"],
            "expected": t["expected"],
            "output": res.get("output"),
            "status": res.get("status"),
            "error": res.get("error"),
            "match": res.get("output") == t["expected"]
        })

    return results


# =========================
# FAILED CASES
# =========================
def get_failed_cases(results):
    return [
        {
            "input": r["input"],
            "expected": r["expected"],
            "output": r["output"],
            "error": r["error"]
        }
        for r in results if not r["match"]
    ]


# =========================
# OPTIMIZER
# =========================
def optimize_solution(code, description, constraints, failed_cases):

    prompt = f"""
You are a competitive programming optimizer.

GOAL:
Improve time and space complexity while preserving correctness.

RULES:
- Output ONLY Python code
- Must define class Solution
- No explanations
- Do NOT remove correctness

PROBLEM:
{description}

CONSTRAINTS:
{constraints}

CURRENT CODE:
{code}

FAILED CASES:
{json.dumps(failed_cases, indent=2)}

OUTPUT:
Return ONLY valid Python class Solution.
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return clean_code(res["message"]["content"])


# =========================
# FULL PIPELINE
# =========================
def full_pipeline(base_solution, description, constraints, tests, validated_tests):

    print("\n================ PIPELINE START ================\n")

    if not is_valid_code(base_solution):
        raise ValueError("Invalid base solution")

    print("\nBASE SOLUTION:\n")
    print(base_solution)



    # STEP 2: VALIDATION ON BASE
    val_results_base = run_tests(base_solution, validated_tests)
    val_pass_base = sum(r["match"] for r in val_results_base)

    print(f"BASE VALIDATION: {val_pass_base}/{len(val_results_base)}")

    # STEP 3: FAILURES
    failed_cases = get_failed_cases( val_results_base)

    print("\nFAILED CASES:")
    print(json.dumps(failed_cases, indent=2))

    # STEP 4: OPTIMIZE
    optimized_code = optimize_solution(
        base_solution,
        description,
        constraints,
        failed_cases
    )

    print("\nOPTIMIZED CODE:\n")
    print(optimized_code)

    if not is_valid_code(optimized_code):
        print("❌ Invalid optimized → fallback")
        return base_solution

    # STEP 5: TEST OPTIMIZED
    opt_base_results = run_tests(optimized_code, tests)
    opt_val_results = run_tests(optimized_code, validated_tests)

    opt_base_pass = sum(r["match"] for r in opt_base_results)
    opt_val_pass = sum(r["match"] for r in opt_val_results)

    print(f"\nOPT BASE: {opt_base_pass}/{len(opt_base_results)}")
    print(f"OPT VAL: {opt_val_pass}/{len(opt_val_results)}")

    # STEP 6: ACCEPT ONLY IF NOT WORSE
    if opt_val_pass < val_pass_base:
        print("❌ Optimization worse → revert")
        final_code = base_solution
    else:
        final_code = optimized_code

    print("\n================ FINAL CODE ================\n")
    print(final_code)

    print("\n================ PIPELINE END ================\n")

    return final_code

In [303]:
final_code = full_pipeline(
    base_solution=code,
    description=description,
    constraints=constraints,
    tests=tests,
    validated_tests=generated_tests_2
)


================ PIPELINE START ================


BASE SOLUTION:

class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        return [self.find_first(nums, target), self.find_last(nums, target)]

    def find_first(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] < target:
                left = mid + 1
            else:
                right = mid
        return left if left < len(nums) and nums[left] == target else -1

    def find_last(self, nums: List[int], target: int) -> int:
        left, right = 0, len(nums)
        while left < right:
            mid = (left + right) // 2
            if nums[mid] <= target:
                left = mid + 1
            else:
                right = mid
        return right - 1 if right > 0 and nums[right - 1] == target else -1
BASE VALIDATION: 0/16

FAILED CASES:
[
  {
    "input": "nums = [5

## Tutoring with detailed differences

In [296]:
import ollama

MODEL_NAME = "llama3"


def llm_compare_solutions(base_code: str, optimized_code: str, description: str):
    prompt = f"""
You are a senior programming tutor and code reviewer.

Your task:
Compare BASE and OPTIMIZED solutions line-by-line and explain improvements.

You must:
- Explain differences in simple but precise terms
- Identify correctness improvements
- Identify performance/complexity changes (if any)
- Say if optimization is REAL or just REFACTORING
- Be strict: do NOT assume improvement unless it exists

========================
PROBLEM DESCRIPTION
========================
{description}

========================
BASE SOLUTION
========================
{base_code}

========================
OPTIMIZED SOLUTION
========================
{optimized_code}

========================
OUTPUT FORMAT
========================

1. OVERALL SUMMARY
- explain how you planned to solve the problem
- What both solutions do
- Whether optimized is truly better or not

2. LINE-BY-LINE DIFFERENCE
For each meaningful change:
- What changed
- Why it changed
- Impact on correctness or efficiency

3. LOGIC ANALYSIS
- Are both logically equivalent?
- Any bug fixes introduced?

4. COMPLEXITY COMPARISON
- Time complexity (base vs optimized)
- Space complexity

5. FINAL VERDICT
- "True Optimization" OR "Refactor Only" OR "Regression Risk"

Keep it concise, technical, and tutor-like.
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return res["message"]["content"]

report = llm_compare_solutions(
    code,          # base solution
    final_code,    # optimized/final solution
    description
)

print(report)

print('---------------------------- base solution ----------------------------------')
print(code)

print('---------------------------- optimized solution -----------------------------')
print(final_code)

**OVERALL SUMMARY**

The problem is to find the first and last position of an element in a sorted array. Both solutions use binary search to achieve O(log n) runtime complexity.

The base solution uses two separate functions for finding the first and last positions, while the optimized solution is identical to the base solution.

Upon closer inspection, I found that the optimized solution is actually **REFORMATTING ONLY**, as it does not introduce any significant improvements or bug fixes. The logic remains unchanged.

**LINE-BY-LINE DIFFERENCE**

None, as both solutions are identical.

**LOGIC ANALYSIS**

Both solutions are logically equivalent and do not introduce any bug fixes.

**COMPLEXITY COMPARISON**

Time complexity: Both solutions have O(log n) time complexity.
Space complexity: Both solutions have O(1) space complexity (constant).

**FINAL VERDICT**

"REFORMATTING ONLY". The optimized solution does not introduce any significant improvements or bug fixes, and the logic remains